In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce.bronze.ingestion_audit (
    run_id STRING,
    source_file STRING,
    target_table STRING,
    rows_read BIGINT,
    ingestion_timestamp TIMESTAMP,
    status STRING,
    error_message STRING
)
USING DELTA
""")

In [0]:
spark.sql("DESCRIBE TABLE ecommerce.bronze.ingestion_audit").show()

In [0]:
import re
import uuid

from pyspark.sql.functions import current_timestamp

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)


# --------------------------------------------------
# Configuration
# --------------------------------------------------

BASE_PATH = "/Volumes/ecommerce/bronze/raw_files"
AUDIT_TABLE = "ecommerce.bronze.ingestion_audit"


# --------------------------------------------------
# Helper function: write audit record
# --------------------------------------------------

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("source_file", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("rows_read", LongType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

def write_audit(
    run_id,
    source_file,
    target_table,
    rows_read,
    status,
    error_message=None
):

    audit_df = spark.createDataFrame(
        [(
            run_id,
            source_file,
            target_table,
            rows_read,
            status,
            error_message
        )],
        schema=audit_schema
    ).withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )

    audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)


# --------------------------------------------------
# Ingestion
# --------------------------------------------------

datasets = dbutils.fs.ls(BASE_PATH)

run_id = str(uuid.uuid4())

for file in datasets:

    file_name = file.name

    if not file_name.endswith(".csv"):
        continue

    # Create clean table name
    table_name = file_name.replace(".csv", "")
    table_name = re.sub(r"^olist_", "", table_name)
    table_name = re.sub(r"_dataset$", "", table_name)

    target_table = f"ecommerce.bronze.{table_name}"

    print(f"Processing: {file_name}")

    try:

        # Read CSV
        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(file.path)
        )

        # Trigger Spark execution
        row_count = df.count()

        # Write Bronze table
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )

        # Record successful ingestion
        write_audit(
            run_id=run_id,
            source_file=file_name,
            target_table=target_table,
            rows_read=row_count,
            status="SUCCESS"
        )

        print(
            f"Completed: {table_name} | "
            f"Rows: {row_count}"
        )

    except Exception as e:

        print(f"FAILED: {file_name}")
        print(str(e))

        # Record failed ingestion
        write_audit(
            run_id=run_id,
            source_file=file_name,
            target_table=target_table,
            rows_read=0,
            status="FAILED",
            error_message=str(e)
        )

In [0]:
%sql
SELECT * FROM ecommerce.bronze.ingestion_audit

In [0]:
display(
    spark.table("ecommerce.bronze.ingestion_audit")
    .orderBy("ingestion_timestamp", ascending=False)
)

In [0]:
tables = spark.sql("SHOW TABLES IN ecommerce.bronze")

display(tables)